# Vérification de l'appariement

Les DPE rattachés aux ventes sont-ils les bons ? Trois contrôles :

1. l'écart de surface entre la vente et le DPE retenu ;
2. le délai entre le DPE et la vente ;
3. l'écart de prix par étiquette, avec l'appariement complet et avec un appariement strict (sans les ventes ambiguës et sans les ventes appariées par la distance). C'est la vérification prévue dans la partie 3 du rapport.

## Paramètres

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RACINE))
from src import chargement, config

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.2f}".format)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.spines.top": False, "axes.spines.right": False})

ORDRE = chargement.ETIQUETTES
COULEURS = {"A": "#009C6D", "B": "#52B153", "C": "#78BD76", "D": "#F4E70F",
            "E": "#F0B40F", "F": "#EB8235", "G": "#D7221F"}

DEPARTEMENT = None  # None = France entière ; "44" pour un essai sur un département
TOLERANCE = config.TOLERANCE_SURFACE

In [ ]:
ventes = chargement.charger_ventes(DEPARTEMENT)
appariees = chargement.echantillon_analyse(ventes)
strict = chargement.appariement_strict(appariees)
print(f"{len(appariees):,} ventes dans l'échantillon d'analyse, dont {len(strict):,} en appariement strict")
display(appariees["methode_appariement"].value_counts().to_frame("n"))

## 1. Écart de surface

`ecart_surface` = |surface de la vente - surface du DPE| / surface du DPE. Si l'appariement choisissait un DPE un peu au hasard dans la tolérance, les écarts seraient étalés jusqu'à 10 % (appartements) ou 20 % (maisons). S'il est bon, ils sont concentrés près de zéro.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (type_bien, tol) in zip(axes, TOLERANCE.items()):
    e = appariees.loc[appariees["type_bien"] == type_bien, "ecart_surface"] * 100
    ax.hist(e, bins=np.arange(0, tol * 100 + 0.5, 0.5), color="#14213D")
    ax.axvline(tol * 100, color="#A23B3B", ls="--")
    ax.set(title=f"{type_bien} (n = {len(e):,})", xlabel="écart de surface (%)", ylabel="ventes")
plt.tight_layout()
plt.show()

seuils = [0.005, 0.01, 0.03, 0.05]
pd.DataFrame({f"<= {s:.1%}": appariees.groupby("type_bien", observed=True)["ecart_surface"]
              .apply(lambda x: (x <= s).mean() * 100) for s in seuils}).round(1)

## 2. Délai entre le DPE et la vente

Le DPE doit précéder la vente, et on s'est limité à trois ans.

In [ ]:
ax = (appariees["delai_jours"] / 30.4).plot(kind="hist", bins=36, color="#14213D")
ax.set(xlabel="délai DPE -> vente (mois)", ylabel="ventes", title="Ancienneté du DPE au moment de la vente")
plt.show()
appariees["delai_jours"].describe(percentiles=[0.25, 0.5, 0.75, 0.9]).to_frame("jours").T

## 3. Écart de prix par étiquette selon l'appariement

On compare des ventes semblables : même commune, même type de bien, même tranche de surface, même période de construction et même trimestre de vente. Dans chaque groupe, on mesure l'écart de chaque vente au prix médian du groupe (en logarithme), puis on fait la moyenne par étiquette. Les groupes de moins de cinq ventes sont écartés. Le résultat est exprimé en % par rapport à la classe D.

C'est une première version du calcul, qui sera reprise dans le module des écarts.

In [ ]:
TRANCHES = [0, 25, 40, 60, 80, 100, 150, 400]
CLE = ["code_commune", "type_bien", "tranche_surface", "periode_construction", "trimestre"]

def ecarts(df, groupes=True, min_groupe=5, reference="D"):
    """Écart moyen de prix au m² de chaque étiquette par rapport à `reference`, en %."""
    df = df.dropna(subset=["prix_m2", "etiquette_dpe"]).copy()
    lp = np.log(df["prix_m2"])
    if groupes:
        df["tranche_surface"] = pd.cut(df["surface"], TRANCHES)
        g = df.groupby(CLE, observed=True)
        taille = g["prix_m2"].transform("size")
        lp = (lp - lp.groupby([df[c] for c in CLE], observed=True).transform("median")).where(taille >= min_groupe)
    moy = lp.groupby(df["etiquette_dpe"], observed=True).mean()
    return ((np.exp(moy - moy[reference]) - 1) * 100).reindex(ORDRE), lp.notna().mean()

variantes = {
    "toutes, sans correction": (appariees, False),
    "toutes, à logement comparable": (appariees, True),
    "sans ventes ambiguës": (appariees[~appariees["ambigu"]], True),
    "adresse seulement": (appariees[appariees["methode_appariement"] == "adresse"], True),
    "appariement strict": (strict, True),
}
lignes, part = {}, {}
for nom, (df, g) in variantes.items():
    lignes[nom], part[nom] = ecarts(df, g)
tab = pd.DataFrame(lignes).T
tab.insert(0, "n", [len(df) for df, _ in variantes.values()])
tab.insert(1, "% conservé", (pd.Series(part) * 100).round(1))
tab.round(1)

In [ ]:
fig, ax = plt.subplots()
for nom, ligne in tab.drop(columns=["n", "% conservé"]).iterrows():
    ax.plot(ORDRE, ligne.values, marker="o", label=nom, ls="--" if "sans correction" in nom else "-")
ax.axhline(0, color="grey", lw=0.8)
ax.set(ylabel="écart de prix au m² par rapport à D (%)", title="Écart par étiquette selon l'appariement")
ax.legend(frameon=False, fontsize=8)
plt.show()

## À retenir pour le rapport

À compléter une fois les résultats lus :

- part des paires avec un écart de surface inférieur à 1 % : … ;
- ancienneté médiane du DPE au moment de la vente : … ;
- écart des logements G par rapport à D, appariement complet : … ; appariement strict : … .